In [1]:
# Cella 1

import subprocess
import sys

# Disinstalliamo torchao per evitare il conflitto di versione con PEFT
print("🧹 Rimozione versione obsoleta di torchao...")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], capture_output=True)

packages = [
    "transformers>=4.44.0",
    "trl>=0.9.6",
    "peft>=0.11.0",
    "accelerate>=0.33.0",
    "datasets>=2.20.0",
    "einops",
    "pyarrow",
    "bitsandbytes>=0.46.1",
]

print("📦 Installazione/Aggiornamento pacchetti richiesti...")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + packages
)
print("✅ Dipendenze installate correttamente!")

🧹 Rimozione versione obsoleta di torchao...
📦 Installazione/Aggiornamento pacchetti richiesti...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 107.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 98.4 MB/s eta 0:00:00
✅ Dipendenze installate correttamente!


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

In [2]:
# Cella 2

import os

# Forza PyTorch a vedere solo la prima GPU, eliminando i conflitti multi-GPU di Kaggle
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

BASE_DIR = "/kaggle/working"
OUTPUT_DIR = os.path.join(BASE_DIR, "qwen05b-dpo")
CACHE_DIR = os.path.join(BASE_DIR, "hf_cache")

for d in [OUTPUT_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

os.environ["HF_HOME"] = CACHE_DIR
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"📁 Output  → {OUTPUT_DIR}")
print(f"📁 Cache   → {CACHE_DIR}")

📁 Output  → /kaggle/working/qwen05b-dpo
📁 Cache   → /kaggle/working/hf_cache


In [3]:
# Cella 3

import torch

# Verifichiamo la disponibilità della GPU
print(f"CUDA disponibile : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU utilizzata   : {props.name} — {props.total_memory / 1024**3:.1f} GB VRAM")

# CONFIGURAZIONE MODELLO E DATASET
MODEL_ID     = "Qwen/Qwen2.5-0.5B-Instruct"
DATASET_PATH = "/kaggle/input/datasets/lorenzosalis/autobench-dpo-dataset/dpo_dataset_qwen.parquet"

# IPERPARAMETRI DI TRAINING (Ottimizzati per Qwen 0.5B su T4)
NUM_EPOCHS        = 3
BATCH_SIZE        = 1        
GRAD_ACCUM        = 16        
LEARNING_RATE     = 1e-5      
MAX_LENGTH        = 512
BETA              = 0.1
WARMUP_RATIO      = 0.1     

# LORA
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05

print("✅ Configurazione caricata")

CUDA disponibile : True
GPU utilizzata   : Tesla T4 — 14.6 GB VRAM
✅ Configurazione caricata


In [4]:
# Cella 4

from datasets import load_dataset

# Caricamento diretto del file Parquet generato in precedenza
ds_raw = load_dataset("parquet", data_files=DATASET_PATH, split="train")

# Suddivisione in Train ed Evaluation (95% / 5% dato l'alto numero di righe)
split = ds_raw.train_test_split(test_size=0.05, seed=42)
ds_train = split["train"]
ds_eval  = split["test"]

print(f"Esempi di addestramento (Train) : {len(ds_train)}")
print(f"Esempi di valutazione (Eval)   : {len(ds_eval)}")

Generating train split: 0 examples [00:00, ? examples/s]

Esempi di addestramento (Train) : 20774
Esempi di valutazione (Eval)   : 1094


In [5]:
# Cella 5

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    cache_dir=CACHE_DIR,
    trust_remote_code=True,
)

# Qwen2.5 ha già i pad token impostati, ma applichiamo una verifica di sicurezza
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# DPO richiede padding a destra per evitare disallineamenti nelle log-probabilità
tokenizer.padding_side = "right"

print(f"Vocab size   : {tokenizer.vocab_size}")
print(f"Pad token    : {tokenizer.pad_token!r} (ID: {tokenizer.pad_token_id})")
print(f"Padding side : {tokenizer.padding_side}")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size   : 151643
Pad token    : '<|endoftext|>' (ID: 151643)
Padding side : right


In [6]:
# Cella 6

import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,   # ← aggiunto
    device_map={"": 0},
    cache_dir=CACHE_DIR,
    trust_remote_code=True,
    # dtype=torch.float16,            # ← rimosso: incompatibile con load_in_4bit
    attn_implementation="sdpa",
)
model.config.use_cache = False

print("✅ Modello caricato in FP16 nativo su GPU 0")
print("ℹ️ Il modello di riferimento (ref_model) sulla GPU 1 è stato rimosso per ottimizzare la memoria.")
print("   DPOTrainer con LoRA utilizzerà la disattivazione dinamica degli adapter su una sola GPU.")

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Modello caricato in FP16 nativo su GPU 0
ℹ️ Il modello di riferimento (ref_model) sulla GPU 1 è stato rimosso per ottimizzare la memoria.
   DPOTrainer con LoRA utilizzerà la disattivazione dinamica degli adapter su una sola GPU.


In [7]:
# Cella 7

from peft import LoraConfig, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

print("✅ Configurazione LoRA definita (sarà applicata automaticamente dal DPOTrainer)")

✅ Configurazione LoRA definita (sarà applicata automaticamente dal DPOTrainer)


In [ ]:
# Cella 8

import torch
from trl import DPOTrainer, DPOConfig

# Svuota la cache della GPU per partire con la massima memoria libera
torch.cuda.empty_cache()

dpo_config = DPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, # 4
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM, # 4
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_RATIO,             # 0.1
    fp16=False,                             
    bf16=False,
    optim="paged_adamw_8bit",                   
    gradient_checkpointing=True,
    eval_strategy="steps",
    eval_steps=100,                        
    save_strategy="steps",
    save_steps=200,
    save_total_limit=1,
    logging_steps=10,
    report_to="none",
    remove_unused_columns=False,
    beta=BETA,
    max_length=MAX_LENGTH,                 # 1024
    truncation_mode="keep_start",
    loss_type="sigmoid",
    disable_dropout=True,
    
    # ACCELERAZIONE CPU E RISPARMIO VRAM:
    dataset_num_proc=4,                    # ← CHIAVE: usa 4 core CPU in parallelo per la tokenizzazione (riduce il tempo da ore a minuti!)
    precompute_ref_log_probs=True,         # ← CHIAVE: elimina il modello di riferimento dalla VRAM durante il training
    precompute_ref_batch_size=4,           # Mantiene l'allocazione dei logits del precalcolo leggerissima
)

trainer = DPOTrainer(
    model=model,                           # Modello base grezzo
    ref_model=None,                        # Nessun modello di riferimento duplicato
    peft_config=lora_config,               # Configurazione LoRA
    args=dpo_config,
    train_dataset=ds_train,
    eval_dataset=ds_eval,
    processing_class=tokenizer,
)

print("🚀 Avvio fase di precalcolo veloce su 4 CPU core e avvio training DPO...")
train_result = trainer.train()

print("\n📊 Risultati completati:")
print(train_result)

Tokenizing train dataset (num_proc=4):   0%|          | 0/20774 [00:00<?, ? examples/s]

Tokenizing eval dataset (num_proc=4):   0%|          | 0/1094 [00:00<?, ? examples/s]

Computing reference log probs for train dataset:   0%|          | 0/5194 [00:00<?, ?it/s]

Computing reference log probs for eval dataset:   0%|          | 0/274 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


🚀 Avvio fase di precalcolo veloce su 4 CPU core e avvio training DPO...


Step,Training Loss,Validation Loss


In [ ]:
# Cella 9

import os

FINAL_DIR = os.path.join(OUTPUT_DIR, "final")
os.makedirs(FINAL_DIR, exist_ok=True)

# Salvataggio degli adapter LoRA e delle configurazioni del tokenizer
trainer.model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

# Salvataggio metriche
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()

print(f"\n✅ Adapter LoRA salvato con successo in: {FINAL_DIR}")